# Overview
This notebooks is intended to estimate net worth change for two housing strategies over different time horizons and under different assumptions.

# Notation
## Big Idea
We are tracking net worth over time:

$$
\text{NW}_s(t) = \text{Assets}_s(t) - \text{Liabilities}_s(t)
$$

for $s\in\{ \text{Rent}, \text{Buy}  \}$ with special interest in the quantity:

$$
\Delta(t) = \text{NW}_{buy}(t) - \text{NW}_{rent}(t)
$$

for $t \in \{ 1, 2, 3, 5, 7 \}$ years.

For a subset of the parameter space, we will run Monte Carlo simulations to estimate the distribution of $\Delta(t)$.

# Imports

In [11]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
import math

# Setup

In [4]:
rng = np.random.default_rng(42)

# Parameter Distributions

In [ ]:
@dataclass
class MarginalDist:
    mean: float
    std: float

@dataclass
class StochasticParams:
    appreciation: MarginalDist
    investment_return: MarginalDist
    rent_growth: MarginalDist
    correlation: np.ndarray | None = None

    def means(self) -> np.ndarray:
        return np.array([self.appreciation.mean, self.investment_return.mean, self.rent_growth.mean])

    def covariance_matrix(self) -> np.ndarray:
        stds = np.array([self.appreciation.std, self.investment_return.std, self.rent_growth.std])
        if self.correlation is not None:
            return np.outer(stds, stds) * self.correlation
        return np.diag(stds ** 2)

@dataclass
class DeterministicParams:
    home_price: float
    down_payment: float
    interest_rate: float
    loan_term_years: int
    property_tax_rate: float
    hoa_monthly: float
    insurance_monthly: float
    maintenance_rate: float
    buy_closing_cost_rate: float
    sell_transaction_cost_rate: float
    rent_monthly_initial: float
    initial_savings: float
    monthly_income: float

    # Tax
    ltcg_rate: float
    stcg_rate: float
    home_sale_exclusion: float

# Data Types

In [ ]:
@dataclass
class BuyState:
    home_value: float
    principal_balance: float
    investment_balance: float
    investment_cost_basis: float

@dataclass
class RentState:
    rent: float
    investment_balance: float
    investment_cost_basis: float

def initial_buy_state(det: DeterministicParams) -> BuyState:
    buy_closing_cost = det.home_price * det.buy_closing_cost_rate
    initial_investments = det.initial_savings - det.down_payment - buy_closing_cost

    return BuyState(
        home_value=det.home_price,
        principal_balance=det.home_price - det.down_payment,
        investment_balance=initial_investments,
        investment_cost_basis=initial_investments,
    )

def initial_rent_state(det: DeterministicParams) -> RentState:
    return RentState(
        rent=det.rent_monthly_initial,
        investment_balance=det.initial_savings,
        investment_cost_basis=det.initial_savings,
    )

# Derived Quantities

In [ ]:
def mortgage_payment(det: DeterministicParams) -> float:
    principal = det.home_price - det.down_payment
    monthly_rate = det.interest_rate / 12
    n_payments = det.loan_term_years * 12

    if monthly_rate == 0:
        return principal / n_payments

    return principal * (monthly_rate * (1 + monthly_rate) ** n_payments) / ((1 + monthly_rate) ** n_payments - 1)

def interest_payment(state: BuyState, det: DeterministicParams) -> float:
    monthly_rate = det.interest_rate / 12
    return state.principal_balance * monthly_rate

def principal_payment(state: BuyState, det: DeterministicParams) -> float:
    return mortgage_payment(det) - interest_payment(state, det)

def property_tax(state: BuyState, det: DeterministicParams) -> float:
    return state.home_value * det.property_tax_rate / 12

def maintenance_cost(state: BuyState, det: DeterministicParams) -> float:
    return state.home_value * det.maintenance_rate / 12

def equity(state: BuyState) -> float:
    return state.home_value - state.principal_balance

def sell_transaction_cost(state: BuyState, det: DeterministicParams) -> float:
    return state.home_value * det.sell_transaction_cost_rate

def net_sale_proceeds(state: BuyState, det: DeterministicParams) -> float:
    return state.home_value - state.principal_balance - sell_transaction_cost(state, det)

def owner_monthly_cost(state: BuyState, det: DeterministicParams) -> float:
    return (
        mortgage_payment(det)
        + property_tax(state, det)
        + det.hoa_monthly
        + det.insurance_monthly
        + maintenance_cost(state, det)
    )

def renter_monthly_cost(state: RentState) -> float:
    return state.rent

def investment_tax(balance: float, cost_basis: float, det: DeterministicParams, months_held: int) -> float:
    gain = max(0.0, balance - cost_basis)
    rate = det.ltcg_rate if months_held > 12 else det.stcg_rate
    return gain * rate

def home_sale_tax(state: BuyState, det: DeterministicParams, months_held: int) -> float:
    gain = state.home_value - det.home_price
    if months_held >= 24:
        gain = max(0.0, gain - det.home_sale_exclusion)
    else:
        gain = max(0.0, gain)
    rate = det.ltcg_rate if months_held > 12 else det.stcg_rate
    return gain * rate

def net_worth_rent(state: RentState, det: DeterministicParams, months_held: int) -> float:
    return state.investment_balance - investment_tax(state.investment_balance, state.investment_cost_basis, det, months_held)

def net_worth_buy(state: BuyState, det: DeterministicParams, months_held: int) -> float:
    proceeds = net_sale_proceeds(state, det) - home_sale_tax(state, det, months_held)
    inv_after_tax = state.investment_balance - investment_tax(state.investment_balance, state.investment_cost_basis, det, months_held)
    return proceeds + inv_after_tax

def delta(buy_state: BuyState, rent_state: RentState, det: DeterministicParams, months_held: int) -> float:
    return net_worth_buy(buy_state, det, months_held) - net_worth_rent(rent_state, det, months_held)

def buy_snapshot(state: BuyState, det: DeterministicParams, t: int) -> dict:
    return {
        "t": t,
        "home_value": state.home_value,
        "principal_balance": state.principal_balance,
        "mortgage_payment": mortgage_payment(det),
        "interest_payment": interest_payment(state, det),
        "principal_payment": principal_payment(state, det),
        "property_tax": property_tax(state, det),
        "maintenance_cost": maintenance_cost(state, det),
        "equity": equity(state),
        "sell_transaction_cost": sell_transaction_cost(state, det),
        "home_sale_tax": home_sale_tax(state, det, t),
        "net_sale_proceeds": net_sale_proceeds(state, det) - home_sale_tax(state, det, t),
        "owner_monthly_cost": owner_monthly_cost(state, det),
        "investment_balance_buy": state.investment_balance,
        "investment_tax_buy": investment_tax(state.investment_balance, state.investment_cost_basis, det, t),
        "net_worth_buy": net_worth_buy(state, det, t),
    }

def rent_snapshot(state: RentState, det: DeterministicParams, t: int) -> dict:
    return {
        "rent": state.rent,
        "renter_monthly_cost": renter_monthly_cost(state),
        "investment_balance_rent": state.investment_balance,
        "investment_tax_rent": investment_tax(state.investment_balance, state.investment_cost_basis, det, t),
        "net_worth_rent": net_worth_rent(state, det, t),
    }

## Rate Path Sampling

In [ ]:
@dataclass
class AnnualRates:
    appreciation: float
    investment_return: float
    rent_growth: float

@dataclass
class RatePath:
    years: list[AnnualRates]

    def __len__(self) -> int:
        return len(self.years)

    def __getitem__(self, year_index: int) -> AnnualRates:
        return self.years[year_index]

def sample_rate_paths(
    stochastic: StochasticParams,
    n_years: int,
    n_trials: int,
    rng: np.random.Generator,
) -> list[RatePath]:
    mean = stochastic.means()
    cov = stochastic.covariance_matrix()
    draws = rng.multivariate_normal(mean, cov, size=(n_trials, n_years))

    return [
        RatePath(years=[
            AnnualRates(appreciation=draws[i, y, 0], investment_return=draws[i, y, 1], rent_growth=draws[i, y, 2])
            for y in range(n_years)
        ])
        for i in range(n_trials)
    ]

## Default Scenarios

In [ ]:
DEFAULT_CORRELATION = np.array([
    # appreciation  investment  rent_growth
    [1.0,           0.3,        0.5],
    [0.3,           1.0,        0.2],
    [0.5,           0.2,        1.0],
])

default_stochastic = StochasticParams(
    appreciation=MarginalDist(mean=0.03, std=0.06),
    investment_return=MarginalDist(mean=0.07, std=0.15),
    rent_growth=MarginalDist(mean=0.03, std=0.02),
    correlation=DEFAULT_CORRELATION,
)

default_deterministic = DeterministicParams(
    home_price=500_000,
    down_payment=100_000,
    interest_rate=0.065,
    loan_term_years=30,
    property_tax_rate=0.012,
    hoa_monthly=350,
    insurance_monthly=150,
    maintenance_rate=0.01,
    buy_closing_cost_rate=0.03,
    sell_transaction_cost_rate=0.06,
    rent_monthly_initial=2_500,
    initial_savings=150_000,
    monthly_income=8_000,
    ltcg_rate=0.15,
    stcg_rate=0.24,
    home_sale_exclusion=250_000,
)

# Simulation 
## Buy/Rent Monthly Steps

In [ ]:
def annual_to_monthly(rate_annual: float) -> float:
    return (1.0 + rate_annual) ** (1.0 / 12.0) - 1.0

def step_buy(state: BuyState, det: DeterministicParams, rates: AnnualRates) -> BuyState:
    monthly_inv_return = annual_to_monthly(rates.investment_return)
    monthly_appr = annual_to_monthly(rates.appreciation)

    # Amortization
    interest = state.principal_balance * (det.interest_rate / 12)
    principal = mortgage_payment(det) - interest
    new_balance = state.principal_balance - principal

    # Home appreciates each month
    new_home_value = state.home_value * (1 + monthly_appr)

    # Total monthly outflow
    cost = (
        mortgage_payment(det)
        + new_home_value * det.property_tax_rate / 12
        + det.hoa_monthly
        + det.insurance_monthly
        + new_home_value * det.maintenance_rate / 12
    )

    # Investments grow, add income, subtract costs
    net_contribution = det.monthly_income - cost
    new_investments = state.investment_balance * (1 + monthly_inv_return) + net_contribution

    return BuyState(
        home_value=new_home_value,
        principal_balance=new_balance,
        investment_balance=new_investments,
        investment_cost_basis=state.investment_cost_basis + net_contribution,
    )

def step_rent(state: RentState, det: DeterministicParams, rates: AnnualRates, is_year_boundary: bool) -> RentState:
    monthly_inv_return = annual_to_monthly(rates.investment_return)

    # Rent increases annually at year boundaries
    new_rent = state.rent * (1 + rates.rent_growth) if is_year_boundary else state.rent

    # Investments grow, add income, subtract rent
    net_contribution = det.monthly_income - new_rent
    new_investments = state.investment_balance * (1 + monthly_inv_return) + net_contribution

    return RentState(
        rent=new_rent,
        investment_balance=new_investments,
        investment_cost_basis=state.investment_cost_basis + net_contribution,
    )

## Single Trial

In [ ]:
HORIZONS = [1, 2, 3, 5, 7]

def simulate_trial(det: DeterministicParams, path: RatePath) -> list[dict]:
    n_years = len(path)
    n_months = n_years * 12

    buy = initial_buy_state(det)
    rent = initial_rent_state(det)
    horizon_months = {h * 12 for h in HORIZONS if h <= n_years}

    snapshots = []
    for month in range(1, n_months + 1):
        rates = path[(month - 1) // 12]
        buy = step_buy(buy, det, rates)
        rent = step_rent(rent, det, rates, is_year_boundary=(month % 12 == 1 and month > 1))

        if month in horizon_months:
            snapshots.append({
                "horizon_years": month // 12,
                **buy_snapshot(buy, det, month),
                **rent_snapshot(rent, det, month),
                "delta": delta(buy, rent, det, month),
            })

    return snapshots

## Monte Carlo

In [ ]:
def run_monte_carlo(
    det: DeterministicParams,
    stochastic: StochasticParams,
    n_trials: int = 10_000,
    rng: np.random.Generator = rng,
) -> pd.DataFrame:
    n_years = max(HORIZONS)
    paths = sample_rate_paths(stochastic, n_years, n_trials, rng)

    rows = []
    for trial, path in enumerate(paths):
        for snapshot in simulate_trial(det, path):
            snapshot["trial"] = trial
            rows.append(snapshot)

    return pd.DataFrame(rows)